In [1]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

In [11]:
# Please refer to BED file format
# https://en.wikipedia.org/wiki/BED_(file_format)

def load_data(file_name):
    columns = ['chrom', 'chromStart', 'chromEnd', 'name']
    df = pd.read_csv(file_name, sep="\t", header=None, names=columns)
    return df

In [12]:
h3k4me3_df = load_data("dataset/HepG2_Male.histone.H3K4me3.peak.bed")
h3k9ac_df = load_data("dataset/HepG2_Male.histone.H3K9ac.peak.bed")
h3k27ac_df = load_data("dataset/HepG2_Male.histone.H3K27ac.peak.bed")
h3k27me3_df = load_data("dataset/HepG2_Male.histone.H3K27me3.peak.bed")
h3k9me3_df = load_data("dataset/HepG2_Male.histone.H3K9me3.peak.bed")

In [13]:
frames = [h3k4me3_df, h3k9ac_df, h3k27ac_df, h3k27me3_df, h3k9me3_df]
histone_df = pd.concat(frames)
histone_df.head()

,chrom,chromStart,chromEnd,name
0,chr10,119808,119954,chr10_173
1,chr10,119956,120102,chr10_174
2,chr10,122100,122246,chr10_185
3,chr10,122308,122454,chr10_186
4,chr10,180346,180492,chr10_489


In [16]:
histone_df.describe()

,chromStart,chromEnd
count,3.190860e+05,3.190860e+05
mean,7.568847e+07,7.568862e+07
std,5.573188e+07,5.573188e+07
min,6.893000e+03,7.039000e+03
25%,3.228027e+07,3.228041e+07
50%,6.304105e+07,6.304120e+07
75%,1.112144e+08,1.112145e+08
max,2.492390e+08,2.492392e+08


In [2]:
username = "root"
password = ""
port = 3306
database = "hg38"

In [3]:
engine = create_engine('mysql+pymysql://%s@localhost:%i/%s' %(username, port, database))

In [5]:
sql = "SELECT * FROM ncbirefseq"
df = pd.read_sql_query(sql, engine)

df.head()

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'"
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'"
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'"
3,585,XR_007065314.1,chr1,+,29773,35418,35418,35418,3,"b'29773,30975,34167,'","b'30667,31093,35418,'",0,MIR1302-2HG,none,none,"b'-1,-1,-1,'"
4,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'"


In [6]:
df.shape

(196097, 16)

In [7]:
refseq_df = df[["bin", "name", "chrom", "strand", "txStart", "txEnd"]]
refseq_df.head()

,bin,name,chrom,strand,txStart,txEnd
0,585,NR_046018.2,chr1,+,11873,14409
1,585,NR_024540.1,chr1,-,14361,29370
2,585,NR_106918.1,chr1,-,17368,17436
3,585,XR_007065314.1,chr1,+,29773,35418
4,585,NR_036051.1,chr1,+,30365,30503


In [19]:
def get_tss(row):
    if row['strand'] == "+":
        return row['txStart']
    return row['txEnd']

In [20]:
refseq_df["tss"] = refseq_df.apply(lambda row: get_tss(row), axis=1)
refseq_df.head()

C:\Users\abdul\AppData\Local\Temp\ipykernel_22596\4010847812.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  refseq_df["tss"] = refseq_df.apply(lambda row: get_tss(row), axis=1)


,bin,name,chrom,strand,txStart,txEnd,tss
0,585,NR_046018.2,chr1,+,11873,14409,11873
1,585,NR_024540.1,chr1,-,14361,29370,29370
2,585,NR_106918.1,chr1,-,17368,17436,17436
3,585,XR_007065314.1,chr1,+,29773,35418,29773
4,585,NR_036051.1,chr1,+,30365,30503,30365


In [25]:
refseq_df.describe()

,bin,txStart,txEnd,tss
count,196097.000000,1.960970e+05,1.960970e+05,1.960970e+05
mean,712.968011,6.827849e+07,6.835499e+07,6.831565e+07
std,577.773176,5.751824e+07,5.752883e+07,5.752260e+07
min,0.000000,0.000000e+00,2.950000e+02,0.000000e+00
25%,146.000000,2.162809e+07,2.166936e+07,2.166936e+07
50%,642.000000,5.325209e+07,5.332807e+07,5.332065e+07
75%,1076.000000,1.050254e+08,1.050706e+08,1.050351e+08
max,2484.000000,2.489139e+08,2.489302e+08,2.489139e+08


In [26]:
tss_test = 29370

hist_df = histone_df[(histone_df["chromStart"] >= tss_test - 2000) & 
                     (histone_df["chromStart"] <= tss_test + 2000)]

hist_df.head()

,chrom,chromStart,chromEnd,name
23544,chr17,30588,30734,chr17_149
76691,chr9,28898,29044,chr9_13
21369,chr17,30193,30339,chr17_148
68485,chr9,28898,29044,chr9_13
85644,chr9,28898,29044,chr9_13
